# Chapter 41: Anomaly Detection

Synthetic NRG batch measurements connect anomaly scores with limited investigation capacity.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
sys.path.insert(0,str(Path.cwd().parents[1]/'src'))
from datasciencebook.anomaly import robust_fit,robust_scores,fit_detector,anomaly_scores,select_for_review,precision_at_capacity
print('Imports ready.')


Imports ready.


In [ ]:
rng=np.random.default_rng(41);n_train=300;n_new=100
train=rng.multivariate_normal([100,7.0,2.0],[[36,1.2,0],[1.2,.09,0],[0,0,.04]],n_train)
new=rng.multivariate_normal([100,7.0,2.0],[[36,1.2,0],[1.2,.09,0],[0,0,.04]],n_new);truth=np.zeros(n_new,int)
idx=np.array([8,24,51,73,92]);new[idx]+=np.array([30,-1.2,1.0]);truth[idx]=1
print(f'Reference batches: {n_train}; new batches: {n_new}; injected cases: {truth.sum()}')


Reference batches: 300; new batches: 100; injected cases: 5


In [ ]:
centre,scale=robust_fit(train);rscore=robust_scores(new,centre,scale)
model=fit_detector(IsolationForest(n_estimators=200,contamination=.05),train);iscore=anomaly_scores(model,new)
for capacity in [3,5,10]:
 selected=select_for_review(iscore,capacity);print(f'capacity={capacity} true_cases={truth[selected].sum()} precision={precision_at_capacity(truth,selected):.2f}')


capacity=3 true_cases=3 precision=1.00
capacity=5 true_cases=5 precision=1.00
capacity=10 true_cases=5 precision=0.50


In [ ]:
selected=select_for_review(iscore,5)
for rank,i in enumerate(selected,1): print(f'rank={rank} batch={i} model_score={iscore[i]:.3f} robust_score={rscore[i]:.2f} confirmed={truth[i]}')


rank=1 batch=8 model_score=0.749 robust_score=5.25 confirmed=1
rank=2 batch=24 model_score=0.749 robust_score=5.09 confirmed=1
rank=3 batch=51 model_score=0.749 robust_score=7.07 confirmed=1
rank=4 batch=73 model_score=0.749 robust_score=6.88 confirmed=1
rank=5 batch=92 model_score=0.749 robust_score=6.10 confirmed=1


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(10,4))
axes[0].hist(iscore[truth==0],bins=18,alpha=.7,label='ordinary');axes[0].hist(iscore[truth==1],bins=8,alpha=.8,label='confirmed');axes[0].legend();axes[0].set(xlabel='Anomaly score',ylabel='Batches',title='Score overlap')
axes[1].scatter(new[:,0],new[:,1],c=iscore,cmap='magma',s=28);axes[1].set(xlabel='Fill weight',ylabel='pH',title='Context for investigation')
fig.tight_layout();plt.show()


## Interpretation

The detector ranks cases; it does not diagnose causes. Capacity changes the reviewed set and its precision. Production evaluation also needs delayed outcomes, missed-event costs, alert latency, subgroup checks, and feedback from investigators.


In [ ]:
# Practice: compare the isolation score with the robust univariate score.
